# 07 — Ensemble de modelos

Combina las probabilidades de múltiples modelos fine-tuneados para mejorar el F1.

**Requisitos**: tener los checkpoints `*_best.pt` en `../models/` (descargar de Kaggle si es necesario).

```
models/
  BETO_best.pt
  mDeBERTa_best.pt
  MarIA_best.pt
```

El ensemble:
1. Carga cada checkpoint y corre inferencia en dev y test
2. Combina probabilidades con pesos proporcionales al dev F1
3. Busca los mejores umbrales conjuntos (greedy + Optuna)
4. Guarda `submissions/run_ensemble.csv`

In [1]:
import os, sys, gc, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import f1_score, classification_report
warnings.filterwarnings('ignore')

# ── Rutas ──────────────────────────────────────────────────────────────────
IN_KAGGLE = os.path.exists('/kaggle/input')
if IN_KAGGLE:
    BASE      = '/kaggle/input/datasets/davidreyesales/hisemotions-2026'
    MODELS_DIR = '/kaggle/input/hisemotions-models'  # dataset con los .pt
    OUT_DIR   = '/kaggle/working/submissions'
    SRC_DIR   = '/kaggle/input/datasets/davidreyesales/hisemotions-2026'
else:
    BASE       = '..'
    MODELS_DIR = '../models'
    OUT_DIR    = '../submissions'
    SRC_DIR    = '../src'

sys.path.insert(0, SRC_DIR)
os.makedirs(OUT_DIR, exist_ok=True)

from dataset import EmotionDataset, load_split, EMOTION_COLS
from model import MultiLabelEmotionClassifier

# ── Device ─────────────────────────────────────────────────────────────────
if torch.cuda.is_available():           DEVICE = torch.device('cuda')
elif torch.backends.mps.is_available(): DEVICE = torch.device('mps')
else:                                   DEVICE = torch.device('cpu')
print(f'Device: {DEVICE} | Kaggle: {IN_KAGGLE}')

Device: mps | Kaggle: False


In [2]:
# ── Modelos a ensamblar ────────────────────────────────────────────────────
# Cada entrada: (nombre, hf_model_name, checkpoint_path)
# Ajusta los paths si los archivos tienen otros nombres
ENSEMBLE_MODELS = [
    ('BETO',     'dccuchile/bert-base-spanish-wwm-cased', os.path.join(MODELS_DIR, 'BETO_best.pt')),
    ('mDeBERTa', 'microsoft/mdeberta-v3-base',           os.path.join(MODELS_DIR, 'mDeBERTa_best.pt')),
    ('MarIA',    'IsGarrido/roberta-base-bne',           os.path.join(MODELS_DIR, 'MarIA_best.pt')),
]

CFG = {
    'max_length': 256,
    'batch_size': 16,
    'dropout':    0.3,
}

# Verificar cuáles checkpoints existen
available = [(n, m, p) for n, m, p in ENSEMBLE_MODELS if os.path.exists(p)]
missing   = [(n, p)    for n, _, p in ENSEMBLE_MODELS if not os.path.exists(p)]

print(f'Checkpoints disponibles: {[n for n,_,_ in available]}')
if missing:
    print(f'FALTAN: {missing}')
    print('Descarga los .pt de Kaggle y ponlos en', MODELS_DIR)

Checkpoints disponibles: ['BETO', 'mDeBERTa', 'MarIA']


In [3]:
# ── Carga de datos ─────────────────────────────────────────────────────────
dev_df  = load_split(os.path.join(BASE, 'dev/dev.csv')   if not IN_KAGGLE else f'{BASE}/dev.csv')
test_df = load_split(os.path.join(BASE, 'test/test.csv') if not IN_KAGGLE else f'{BASE}/test.csv')

print(f'Dev:  {len(dev_df)} filas')
print(f'Test: {len(test_df)} filas')

Dev:  425 filas
Test: 863 filas


In [4]:
def get_logits(model_name: str, hf_name: str, ckpt_path: str,
               dev_df, test_df) -> dict:
    """Carga modelo + checkpoint y devuelve logits en dev y test."""
    print(f'\n── {model_name} ──────────────────────────────')
    tokenizer = AutoTokenizer.from_pretrained(hf_name)

    dev_loader  = DataLoader(
        EmotionDataset(dev_df,  tokenizer, CFG['max_length']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=0)
    test_loader = DataLoader(
        EmotionDataset(test_df, tokenizer, CFG['max_length']),
        batch_size=CFG['batch_size'], shuffle=False, num_workers=0)

    ckpt  = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
    model = MultiLabelEmotionClassifier(hf_name, dropout=CFG['dropout']).to(DEVICE)
    model.load_state_dict(ckpt['model'])
    model.eval()

    saved_thresholds = ckpt.get('thresholds', np.full(len(EMOTION_COLS), 0.5))

    def run(loader, has_labels):
        logits_list, labels_list = [], []
        with torch.no_grad():
            for batch in loader:
                ids  = batch['input_ids'].to(DEVICE)
                mask = batch['attention_mask'].to(DEVICE)
                tt   = batch.get('token_type_ids')
                if tt is not None: tt = tt.to(DEVICE)
                logits = model(ids, mask, tt)
                logits_list.append(logits.cpu().numpy())
                if has_labels and 'labels' in batch:
                    labels_list.append(batch['labels'].numpy())
        lo = np.vstack(logits_list)
        la = np.vstack(labels_list) if labels_list else None
        return lo, la

    dev_logits,  dev_labels  = run(dev_loader,  has_labels=True)
    test_logits, _           = run(test_loader, has_labels=False)

    # F1 individual en dev con sus propios umbrales
    dev_probs = 1 / (1 + np.exp(-dev_logits))
    dev_preds = (dev_probs >= saved_thresholds).astype(int)
    solo_f1   = f1_score(dev_labels, dev_preds, average='micro', zero_division=0)
    print(f'  Dev F1 (solo, thresholds guardados): {solo_f1:.4f}')

    del model
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

    return {
        'name':             model_name,
        'dev_logits':       dev_logits,
        'dev_labels':       dev_labels,
        'test_logits':      test_logits,
        'solo_dev_f1':      solo_f1,
        'saved_thresholds': saved_thresholds,
    }

In [5]:
# ── Inferencia de todos los modelos disponibles ────────────────────────────
results = []
for name, hf_name, ckpt_path in available:
    r = get_logits(name, hf_name, ckpt_path, dev_df, test_df)
    results.append(r)

print(f'\nModelos cargados: {[r["name"] for r in results]}')
for r in results:
    print(f'  {r["name"]:12s} — Dev F1 solo: {r["solo_dev_f1"]:.4f}')


── BETO ──────────────────────────────


Some weights of BertModel were not initialized from the model checkpoint at dccuchile/bert-base-spanish-wwm-cased and are newly initialized: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  Dev F1 (solo, thresholds guardados): 0.4566

── mDeBERTa ──────────────────────────────


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

Some weights of RobertaModel were not initialized from the model checkpoint at IsGarrido/roberta-base-bne and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

  Dev F1 (solo, thresholds guardados): 0.4688

Modelos cargados: ['BETO', 'mDeBERTa', 'MarIA']
  BETO         — Dev F1 solo: 0.4566
  mDeBERTa     — Dev F1 solo: 0.4410
  MarIA        — Dev F1 solo: 0.4688


In [6]:
# ── Pesos proporcionales al Dev F1 individual ──────────────────────────────
f1_scores = np.array([r['solo_dev_f1'] for r in results])
weights   = f1_scores / f1_scores.sum()

print('Pesos del ensemble:')
for r, w in zip(results, weights):
    print(f'  {r["name"]:12s}: {w:.3f}')

# ── Combinar probabilidades ────────────────────────────────────────────────
dev_labels  = results[0]['dev_labels']  # son iguales para todos

dev_probs_ensemble  = sum(w * (1 / (1 + np.exp(-r['dev_logits'])))
                          for r, w in zip(results, weights))
test_probs_ensemble = sum(w * (1 / (1 + np.exp(-r['test_logits'])))
                          for r, w in zip(results, weights))

# F1 con umbral 0.5 como baseline
baseline_f1 = f1_score(dev_labels, (dev_probs_ensemble >= 0.5).astype(int),
                        average='micro', zero_division=0)
print(f'\nEnsemble Dev F1 (threshold=0.5): {baseline_f1:.4f}')

Pesos del ensemble:
  BETO        : 0.334
  mDeBERTa    : 0.323
  MarIA       : 0.343

Ensemble Dev F1 (threshold=0.5): 0.3953


In [7]:
# ── Búsqueda greedy de umbrales sobre las probs del ensemble ───────────────
def greedy_threshold_search(probs, labels, search_range=None):
    if search_range is None:
        search_range = np.arange(0.05, 0.96, 0.05)
    n_labels = probs.shape[1]
    best_thresholds = np.full(n_labels, 0.5)

    for i in range(n_labels):
        best_t, best_s = 0.5, 0.0
        for t in search_range:
            th = best_thresholds.copy()
            th[i] = t
            s = f1_score(labels, (probs >= th).astype(int),
                         average='micro', zero_division=0)
            if s > best_s:
                best_s, best_t = s, t
        best_thresholds[i] = best_t

    final_f1 = f1_score(labels, (probs >= best_thresholds).astype(int),
                        average='micro', zero_division=0)
    return best_thresholds, final_f1


greedy_thresholds, greedy_f1 = greedy_threshold_search(
    dev_probs_ensemble, dev_labels)

print('Umbrales greedy:')
for col, t in zip(EMOTION_COLS, greedy_thresholds):
    print(f'  {col:10s}: {t:.2f}')
print(f'Ensemble Dev F1 (greedy): {greedy_f1:.4f}')

Umbrales greedy:
  anger     : 0.10
  fear      : 0.85
  joy       : 0.80
  sadness   : 0.80
  surprise  : 0.85
  hope      : 0.30
Ensemble Dev F1 (greedy): 0.4704


In [8]:
# ── Búsqueda conjunta con Optuna (más precisa que greedy) ──────────────────
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'optuna'], check=True)
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

_probs  = dev_probs_ensemble
_labels = dev_labels

def objective(trial):
    thresholds = np.array([
        trial.suggest_float(f't_{e}', 0.05, 0.95) for e in EMOTION_COLS
    ])
    preds = (_probs >= thresholds).astype(int)
    return f1_score(_labels, preds, average='micro', zero_division=0)

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=500, show_progress_bar=True)

best_params     = study.best_params
optuna_thresholds = np.array([best_params[f't_{e}'] for e in EMOTION_COLS])
optuna_f1        = study.best_value

print('\nUmbrales Optuna:')
for col, t in zip(EMOTION_COLS, optuna_thresholds):
    print(f'  {col:10s}: {t:.3f}')
print(f'Ensemble Dev F1 (Optuna): {optuna_f1:.4f}')

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)

[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: /Users/davidreyes/Documents/Proyectos/HISEMOTIONS_2026/.venv/bin/python -m pip install --upgrade pip


  0%|          | 0/500 [00:00<?, ?it/s]


Umbrales Optuna:
  anger     : 0.111
  fear      : 0.877
  joy       : 0.885
  sadness   : 0.816
  surprise  : 0.228
  hope      : 0.078
Ensemble Dev F1 (Optuna): 0.4721


In [9]:
# ── Elige los mejores umbrales ─────────────────────────────────────────────
print(f'Dev F1 — greedy:  {greedy_f1:.4f}')
print(f'Dev F1 — Optuna:  {optuna_f1:.4f}')

if optuna_f1 >= greedy_f1:
    best_thresholds = optuna_thresholds
    print('→ Se usarán umbrales Optuna')
else:
    best_thresholds = greedy_thresholds
    print('→ Se usarán umbrales greedy')

Dev F1 — greedy:  0.4704
Dev F1 — Optuna:  0.4721
→ Se usarán umbrales Optuna


In [10]:
# ── Report por emoción en dev ──────────────────────────────────────────────
dev_preds_final = (dev_probs_ensemble >= best_thresholds).astype(int)

print(classification_report(
    dev_labels, dev_preds_final,
    target_names=EMOTION_COLS, zero_division=0))

final_dev_f1 = f1_score(dev_labels, dev_preds_final, average='micro', zero_division=0)
print(f'Dev Micro F1 final: {final_dev_f1:.4f}')

              precision    recall  f1-score   support

       anger       0.45      0.60      0.51        52
        fear       0.32      0.42      0.37        40
         joy       0.78      0.48      0.60        29
     sadness       0.42      0.64      0.51        70
    surprise       0.12      0.14      0.13         7
        hope       0.34      0.72      0.47        85

   micro avg       0.39      0.60      0.47       283
   macro avg       0.41      0.50      0.43       283
weighted avg       0.42      0.60      0.48       283
 samples avg       0.21      0.30      0.23       283

Dev Micro F1 final: 0.4721


In [11]:
import zipfile

# ── Predicciones sobre test ────────────────────────────────────────────────
test_preds = (test_probs_ensemble >= best_thresholds).astype(int)

# Solo las 6 columnas de emoción (formato Codabench)
submission = pd.DataFrame(test_preds, columns=EMOTION_COLS)

csv_path = os.path.join(OUT_DIR, 'predictions.csv')
zip_path = os.path.join(OUT_DIR, 'predictions.zip')

submission.to_csv(csv_path, index=False)

with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    zf.write(csv_path, 'predictions.csv')

print(f'predictions.csv  → {csv_path}')
print(f'predictions.zip  → {zip_path}  (listo para Codabench)')
print(f'Filas: {len(submission)}')
print('\nDistribución predicciones test:')
print(submission[EMOTION_COLS].sum().sort_values(ascending=False))

predictions.csv  → ../submissions/predictions.csv
predictions.zip  → ../submissions/predictions.zip  (listo para Codabench)
Filas: 863

Distribución predicciones test:
hope        306
sadness     176
anger       113
fear         63
joy          20
surprise     10
dtype: int64


In [12]:
# ── Ablación: cuánto aporta cada modelo al ensemble ────────────────────────
print('Ablación — Dev F1 sin cada modelo:')
for leave_out in results:
    remaining = [r for r in results if r['name'] != leave_out['name']]
    if not remaining:
        continue
    f1_rem = np.array([r['solo_dev_f1'] for r in remaining])
    w_rem  = f1_rem / f1_rem.sum()
    probs_rem = sum(w * (1 / (1 + np.exp(-r['dev_logits'])))
                    for r, w in zip(remaining, w_rem))
    _, f1_rem_val = greedy_threshold_search(probs_rem, dev_labels)
    print(f'  sin {leave_out["name"]:12s}: Dev F1 = {f1_rem_val:.4f}  '
          f'(delta = {f1_rem_val - final_dev_f1:+.4f})')

Ablación — Dev F1 sin cada modelo:
  sin BETO        : Dev F1 = 0.4669  (delta = -0.0051)
  sin mDeBERTa    : Dev F1 = 0.4746  (delta = +0.0025)
  sin MarIA       : Dev F1 = 0.4734  (delta = +0.0014)


In [13]:
from sklearn.metrics import f1_score
import pandas as pd

test_labels_df = pd.read_csv('../test/test.csv').dropna(subset=['text'])
for col in EMOTION_COLS:
    test_labels_df[col] = test_labels_df[col].fillna(0).astype(int)

preds_df = pd.read_csv('../submissions/predictions.csv')

test_f1 = f1_score(
    test_labels_df[EMOTION_COLS].values,
    preds_df[EMOTION_COLS].values,
    average='micro', zero_division=0
)
print(f'Ensemble Test Micro F1: {test_f1:.4f}')

Ensemble Test Micro F1: 0.3919


## Próximos pasos

- Si faltan checkpoints, descárgalos de Kaggle:
  ```
  kaggle datasets download <usuario>/<dataset-checkpoints> -p models/
  ```
- Para añadir más modelos al ensemble, agrégalos a `ENSEMBLE_MODELS` en la celda de configuración.
- Para reentrenar con train+dev y mejorar el F1 final, descomenta la celda `retrain-final` en el notebook 03.